<a href="https://colab.research.google.com/github/AConn03/Long_Exposure/blob/main/Train_Rust_Detection_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Datasets Used**
---
Train On Plants and Landscapes (Clean)

* Hugging Face leaves - 9,760 Images: https://huggingface.co/datasets/walker81131/plantSelected

* Hugging Face landscapes - 9,600 Images: https://huggingface.co/datasets/amaye15/landscapes/viewer/default/train?row=0

* Hugging Face plants - 4,200 Images: https://huggingface.co/datasets/Elapsedf/Plant-Pathology

Train for Rust

* Roboflow - 3,688 Images: https://universe.roboflow.com/corrosionlabneo/corrosion-fuk2x/dataset/2

* KAGGLE rust - 253 Images: https://www.kaggle.com/datasets/nikitanikitos/corrosion-annotated

* KAGGLE rust and clean metal - 182 Images: https://www.kaggle.com/datasets/benpepperpots/rust-iron-dataset

* Hugging Face rust - 750 Images: https://huggingface.co/datasets/BinKhoaLe1812/Corrosion_Rust/viewer/default/train?p=7

* Hugging Face rust - 1,249 Images: https://huggingface.co/datasets/Francesco/corrosion-bi3q3

* GitHub CorrSave (Rust & Clean) - 30 Images: https://github.com/irfixq/CorrSave

Training for Cracks

* Hugging Face cracks - 1,290 Images: https://huggingface.co/datasets/rishitunu/ecc_crackdetector_dataset_exhaustive/viewer/default/train?p=1

* Hugging Face cracks 2 - 273 Images: https://huggingface.co/datasets/aliasghar-j/concrete-crack-dataset

* Virginia Tech Labeled Cracks in the Wild (LCW) - 3,817 Images: https://data.lib.vt.edu/articles/dataset/Labeled_Cracks_in_the_Wild_LCW_Dataset/16624672

* Virginia Tech Conglomerate Concrete Crack (CCC) - ~10,000 Images: https://data.lib.vt.edu/articles/dataset/Concrete_Crack_Conglomerate_Dataset/16625056

Training for Both (Mixed Defects)

* Kaggle Corrosion & Cracks - 4,050 Images: https://www.kaggle.com/datasets/linsangancjm/corrosion-crack

In [ ]:
#@title Part 1: Modular Data Export (Unlimited Datasets)
# ==============================================================================
# CELL 1: DATA PIPELINE
# Downloads full datasets without limits, checks Drive for existing datasets to
# skip them, and processes new ones into Class Folders on Drive.
# ==============================================================================

import os
import shutil
import multiprocessing
from tqdm import tqdm

# Install required libraries
os.system("pip install -q transformers datasets evaluate accelerate torchvision kaggle ninja roboflow")

# --- CONNECT GOOGLE DRIVE ---
from google.colab import drive
if not os.path.exists('/content/drive'):
    print("Mounting Google Drive...")
    drive.mount('/content/drive')

# --- CONFIGURATION ---
DRIVE_BASE = "/content/drive/MyDrive/Colab Notebooks/Capstone/Training Data"
os.makedirs(DRIVE_BASE, exist_ok=True)

# --- SECRETS & CREDENTIALS ---
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    ROBOFLOW_KEY = userdata.get('ROBOFLOW_KEY')
except:
    print("⚠️ Secrets not found. Ensure KAGGLE_USERNAME, KAGGLE_KEY, and ROBOFLOW_KEY are in Colab Secrets.")
    ROBOFLOW_KEY = None

from datasets import load_dataset, concatenate_datasets, Features, Image, ClassLabel, DatasetDict
from huggingface_hub import snapshot_download

num_workers = multiprocessing.cpu_count()
hf_workers = min(2, num_workers)

global_labels = ['clean', 'rust', 'crack']
label2id = {c: i for i, c in enumerate(global_labels)}
id2label = {i: c for i, c in enumerate(global_labels)}
target_features = Features({'image': Image(), 'label': ClassLabel(names=global_labels)})

# ==========================================
# INVENTORY SYSTEM
# ==========================================
def get_installed_sources():
    """Scans Drive folders to see which Source IDs are already present."""
    installed = set()
    for label_folder in ['rust', 'cracks', 'clean']:
        path = os.path.join(DRIVE_BASE, label_folder)
        if os.path.exists(path):
            for f in os.listdir(path):
                if f.endswith('.zip'):
                    source_id = f.split('_')[0]
                    installed.add(source_id)
    return installed

print("🔍 Checking Drive Inventory...")
INSTALLED_SOURCES = get_installed_sources()
if INSTALLED_SOURCES:
    print(f"✅ Found {len(INSTALLED_SOURCES)} datasets already installed: {', '.join(INSTALLED_SOURCES)}")
else:
    print("ℹ️ No datasets found on Drive yet. Starting fresh.")

# ==========================================
# PROCESSING & EXPORT HELPER
# ==========================================
def export_source_to_drive(source_id, train_ds, test_ds, label_name):
    """Zips and moves a specific dataset source to Drive immediately."""
    temp_root = f"/content/temp_{source_id}"
    drive_folder = os.path.join(DRIVE_BASE, label_name)
    os.makedirs(drive_folder, exist_ok=True)

    for split_name, ds in [('train', train_ds), ('test', test_ds)]:
        if ds is None or len(ds) == 0: continue

        split_dir = os.path.join(temp_root, split_name)
        os.makedirs(split_dir, exist_ok=True)

        class_dir = os.path.join(split_dir, label_name)
        os.makedirs(class_dir, exist_ok=True)

        for i, item in enumerate(tqdm(ds, desc=f"📦 Zipping {source_id} ({split_name})", leave=False)):
            try:
                img = item['image'].convert("RGB")
                img.save(os.path.join(class_dir, f"{i}.jpg"), "JPEG")
            except Exception as e:
                continue

        zip_filename = f"{source_id}_{label_name}_{split_name}"
        shutil.make_archive(f"/content/{zip_filename}", 'zip', split_dir)
        shutil.move(f"/content/{zip_filename}.zip", os.path.join(drive_folder, f"{zip_filename}.zip"))

    if os.path.exists(temp_root): shutil.rmtree(temp_root)
    print(f"🚀 Successfully exported {source_id} to {label_name} folder.")

# Helper to process datasets and map labels safely
def standardize_dataset(d, target_label_name):
    if not isinstance(d, DatasetDict):
        d = DatasetDict({'train': d})

    if 'test' not in d.keys() and 'validation' not in d.keys():
        d = d['train'].train_test_split(test_size=0.2, seed=42)

    target_id = label2id[target_label_name]

    def process(split_ds):
        if split_ds is None: return None

        img_col = None
        for col in split_ds.column_names:
            if isinstance(split_ds.features[col], Image):
                img_col = col
                break

        if img_col is None:
            img_col = next((c for c in split_ds.column_names if c.lower() in ["image", "img", "pixel_values", "data"]), split_ds.column_names[0])

        if img_col != 'image':
            split_ds = split_ds.rename_column(img_col, 'image')

        cols_to_remove = [c for c in split_ds.column_names if c != 'image']
        split_ds = split_ds.remove_columns(cols_to_remove)
        split_ds = split_ds.add_column('label', [target_id] * len(split_ds))
        return split_ds.cast(target_features)

    return process(d['train']), process(d.get('test', d.get('validation')))

# ==========================================
# DATASET LOADING BLOCKS
# ==========================================

# --- 1. RUST & CLEAN SOURCES ---
print("\n--- Processing Rust/Metal Datasets ---")

sid = "BinKhoaLe"
if sid not in INSTALLED_SOURCES:
    print(f"-> Loading {sid}...")
    ds = load_dataset("BinKhoaLe1812/Corrosion_Rust", num_proc=hf_workers)
    tr, te = standardize_dataset(ds, 'rust')
    export_source_to_drive(sid, tr, te, 'rust')
else: print(f"⏭️ Skipping {sid} (Already installed)")

sid = "Francesco"
if sid not in INSTALLED_SOURCES:
    print(f"-> Loading {sid}...")
    ds = load_dataset("Francesco/corrosion-bi3q3", num_proc=hf_workers)
    tr, te = standardize_dataset(ds, 'rust')
    export_source_to_drive(sid, tr, te, 'rust')
else: print(f"⏭️ Skipping {sid} (Already installed)")

sid = "BenPepper"
if sid not in INSTALLED_SOURCES:
    print(f"-> Loading {sid}...")
    os.system("kaggle datasets download -d benpepperpots/rust-iron-dataset --unzip -p ./kp1")
    ds = load_dataset("imagefolder", data_dir="./kp1")
    tr, te = standardize_dataset(ds, 'rust')
    export_source_to_drive(sid, tr, te, 'rust')
else: print(f"⏭️ Skipping {sid} (Already installed)")

sid = "NikitaRust"
if sid not in INSTALLED_SOURCES:
    print(f"-> Loading {sid}...")
    os.system("kaggle datasets download -d nikitanikitos/corrosion-annotated --unzip -p ./kp2")
    ds = load_dataset("imagefolder", data_dir="./kp2")
    tr, te = standardize_dataset(ds, 'rust')
    export_source_to_drive(sid, tr, te, 'rust')
else: print(f"⏭️ Skipping {sid} (Already installed)")

sid = "CorrSave"
if f"{sid}Rust" not in INSTALLED_SOURCES:
    print(f"-> Loading {sid} from GitHub...")
    os.system("git clone https://github.com/irfixq/CorrSave.git ./corrsave_repo")
    try:
        os.makedirs("./temp_corrsave_rust/rust", exist_ok=True)
        rust_count = 0
        for root, dirs, files in os.walk("./corrsave_repo/Corroded"):
            for file in files:
                if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                    shutil.copy(os.path.join(root, file), f"./temp_corrsave_rust/rust/r_{rust_count}.jpg")
                    rust_count += 1
        if rust_count > 0:
            ds = load_dataset("imagefolder", data_dir="./temp_corrsave_rust")
            tr, te = standardize_dataset(ds, 'rust')
            export_source_to_drive(f"{sid}Rust", tr, te, 'rust')
    except Exception as e: print(f"Error {sid}: {e}")
else: print(f"⏭️ Skipping {sid} (Already installed)")

# --- 2. CRACK SOURCES ---
print("\n--- Processing Crack Datasets ---")

sid = "ConcreteCrack"
if sid not in INSTALLED_SOURCES:
    print(f"-> Loading {sid}...")
    ds = load_dataset("aliasghar-j/concrete-crack-dataset", num_proc=hf_workers)
    tr, te = standardize_dataset(ds, 'crack')
    export_source_to_drive(sid, tr, te, 'cracks')
else: print(f"⏭️ Skipping {sid} (Already installed)")

sid = "ECC"
if sid not in INSTALLED_SOURCES:
    print(f"-> Loading {sid}...")
    ds = load_dataset("rishitunu/ecc_crackdetector_dataset_exhaustive", num_proc=hf_workers)
    tr, te = standardize_dataset(ds, 'crack')
    export_source_to_drive(sid, tr, te, 'cracks')
else: print(f"⏭️ Skipping {sid} (Already installed)")

sid = "VT_LCW"
if sid not in INSTALLED_SOURCES:
    print(f"-> Loading {sid} from VT...")
    os.system("wget -q https://data.lib.vt.edu/ndownloader/files/30916327 -O /content/lcw.zip")
    os.system("unzip -q /content/lcw.zip -d /content/lcw_data")
    try:
        os.makedirs("./temp_lcw/crack", exist_ok=True)
        img_count = 0
        for root, dirs, files in os.walk("/content/lcw_data"):
            folder_name = os.path.basename(root).lower()
            if folder_name in ['mask', 'masks', 'label', 'labels', 'ground_truth']: continue
            for file in files:
                if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                    if 'mask' in file.lower() or 'label' in file.lower(): continue
                    shutil.copy(os.path.join(root, file), f"./temp_lcw/crack/lcw_{img_count}.jpg")
                    img_count += 1
        ds = load_dataset("imagefolder", data_dir="./temp_lcw")
        tr, te = standardize_dataset(ds, 'crack')
        export_source_to_drive(sid, tr, te, 'cracks')
    except Exception as e: print(f"Error {sid}: {e}")
else: print(f"⏭️ Skipping {sid} (Already installed)")

# --- 3. INDUSTRIAL TEXTURES (CLEAN / NEGATIVES) ---
print("\n--- Processing Metal Textures (Clean) ---")

sid = "NEUMetal"
if sid not in INSTALLED_SOURCES:
    print(f"-> Loading NEU Surface Defect Database from Kaggle...")
    os.system("kaggle datasets download -d kaustubhdikshit/neu-surface-defect-database --unzip -p ./neu_data")
    try:
        os.makedirs("./temp_neu/clean", exist_ok=True)
        img_count = 0
        for root, dirs, files in os.walk("./neu_data"):
            for file in files:
                if file.lower().endswith(('.bmp', '.jpg', '.jpeg')):
                    shutil.copy(os.path.join(root, file), f"./temp_neu/clean/neu_{img_count}.jpg")
                    img_count += 1
        ds = load_dataset("imagefolder", data_dir="./temp_neu")
        tr, te = standardize_dataset(ds, 'clean')
        export_source_to_drive(sid, tr, te, 'clean')
    except Exception as e: print(f"Error {sid}: {e}")
else: print(f"⏭️ Skipping {sid} (Already installed)")

sid = "DTDTextures"
if sid not in INSTALLED_SOURCES:
    print(f"-> Loading DTD Textures via Direct Download (Avoiding HF Hangs)...")
    # Download the whole archive at once to prevent connection stalls
    os.system("wget -q https://www.robots.ox.ac.uk/~vgg/data/dtd/download/dtd-r1.0.1.tar.gz -O /content/dtd.tar.gz")
    os.system("tar -xzf /content/dtd.tar.gz -C /content/")
    try:
        os.makedirs("./temp_dtd/clean", exist_ok=True)
        img_count = 0
        for root, dirs, files in os.walk("/content/dtd/images"):
            for file in files:
                if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                    shutil.copy(os.path.join(root, file), f"./temp_dtd/clean/dtd_{img_count}.jpg")
                    img_count += 1

        if img_count > 0:
            ds = load_dataset("imagefolder", data_dir="./temp_dtd")
            tr, te = standardize_dataset(ds, 'clean')
            export_source_to_drive(sid, tr, te, 'clean')
    except Exception as e: print(f"Error {sid}: {e}")
else: print(f"⏭️ Skipping {sid} (Already installed)")

# --- 4. OTHER NEGATIVES ---
print("\n--- Processing General Negatives ---")

add_negative = lambda sid, path: export_source_to_drive(sid, *standardize_dataset(load_dataset(path), 'clean'), 'clean') if sid not in INSTALLED_SOURCES else print(f"⏭️ Skipping {sid}")

add_negative("PlantsHF", "walker81131/plantSelected")
add_negative("LandscapesHF", "amaye15/landscapes")
add_negative("PathologyHF", "Elapsedf/Plant-Pathology")

print(f"\n✅ ALL DATASETS PROCESSED AND ZIPPED TO DRIVE!")

Mounting Google Drive...
Mounted at /content/drive
🔍 Checking Drive Inventory...
✅ Found 19 datasets already installed: PlantsHF, CCCD, BinKhoaLeRust, BinKhoaLe, NEUMetal, Francesco, KaggleMixedClean, CorrSaveClean, NikitaRust, LandscapesHF, CorrSave, BinKhoaLeClean, CorrSaveRust, PathologyHF, ConcreteCrack, ECC, BenPepper, DTDTextures, roboflow

--- Processing Rust/Metal Datasets ---
⏭️ Skipping BinKhoaLe (Already installed)
⏭️ Skipping Francesco (Already installed)
⏭️ Skipping BenPepper (Already installed)
⏭️ Skipping NikitaRust (Already installed)
⏭️ Skipping CorrSave (Already installed)

--- Processing Crack Datasets ---
⏭️ Skipping ConcreteCrack (Already installed)
⏭️ Skipping ECC (Already installed)
-> Loading VT_LCW from VT...
Error VT_LCW: The directory at /content/temp_lcw doesn't contain any data files

--- Processing Metal Textures (Clean) ---
⏭️ Skipping NEUMetal (Already installed)
⏭️ Skipping DTDTextures (Already installed)

--- Processing General Negatives ---
⏭️ Skippin

In [ ]:
#@title File Sizes
import os

def get_size_format(b, factor=1024, suffix="B"):
    for unit in ["", "K", "M", "G", "T", "P"]:
        if b < factor:
            return f"{b:.2f} {unit}{suffix}"
        b /= factor

def get_directory_size(path):
    total_size = 0
    try:
        with os.scandir(path) as it:
            for entry in it:
                if entry.is_file():
                    total_size += entry.stat().st_size
                elif entry.is_dir():
                    total_size += get_directory_size(entry.path)
    except OSError:
        return 0
    return total_size

# --- EXECUTION ---
print(f"\n📊 Scanning: {DRIVE_BASE}")
if os.path.exists(DRIVE_BASE):
    categories = [d for d in os.listdir(DRIVE_BASE) if os.path.isdir(os.path.join(DRIVE_BASE, d))]
    grand_total = 0

    for cat in categories:
        cat_path = os.path.join(DRIVE_BASE, cat)
        c_size = get_directory_size(cat_path)
        grand_total += c_size
        print(f"📁 {cat.upper()}: {get_size_format(c_size)}")

    print("-" * 30)
    print(f"📦 TOTAL DATASET SIZE: {get_size_format(grand_total)}")

    if grand_total == 0:
        print("⚠️ Warning: Size is 0. Check if 'DRIVE_BASE' path is correct or if files are still syncing.")
else:
    print("❌ Error: DRIVE_BASE path does not exist.")


NameError: name 'DRIVE_BASE' is not defined

In [ ]:
#@title Part 2: Model Training (Optimized for High-End GPUs)
# ==============================================================================
# CELL 2: TRAINING PIPELINE
# Pulls modular zips from Drive folders, safely prevents filename overwriting,
# uses full datasets, applies advanced augmentation, and smoothed weights.
# Dynamically scales batch size and workers based on available VRAM and RAM.
# ==============================================================================

import os
import shutil
import multiprocessing

# --- INSTALL REQUIRED LIBRARIES ---
print("Installing required AI libraries...")
os.system("pip install -q transformers datasets evaluate accelerate torchvision ninja psutil")

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import psutil
from datasets import load_dataset
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer
)
from torchvision.transforms import (
    Compose, Normalize, RandomHorizontalFlip, RandomVerticalFlip,
    RandomResizedCrop, RandomRotation, ColorJitter, ToTensor, Resize, CenterCrop,
    RandomGrayscale, RandomAdjustSharpness
)
import evaluate

# --- CONNECT GOOGLE DRIVE ---
try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        print("Mounting Google Drive...")
        drive.mount('/content/drive')
except:
    print("Not in Colab. Skipping Drive mount.")

# --- CONFIGURATION ---
DRIVE_BASE = "/content/drive/MyDrive/Colab Notebooks/Capstone/Training Data"
LOCAL_TRAIN_DIR = "/content/Local_Training_Data"
DRIVE_MODEL_DIR = "/content/drive/MyDrive/Colab Notebooks/Capstone/models"

print("\n1. Importing Modular Data from Google Drive Class Folders...")
found_zips = False

# Clear out local directory for a clean slate
if os.path.exists(LOCAL_TRAIN_DIR):
    shutil.rmtree(LOCAL_TRAIN_DIR)

for class_folder in ['rust', 'cracks', 'clean']:
    drive_path = os.path.join(DRIVE_BASE, class_folder)
    if not os.path.exists(drive_path): continue

    zip_files = [f for f in os.listdir(drive_path) if f.endswith('.zip')]
    for z in zip_files:
        found_zips = True
        zip_basename = z.replace('.zip', '')
        temp_extract = os.path.join("/content/temp_unpack", zip_basename)
        os.makedirs(temp_extract, exist_ok=True)

        print(f"   [~] Extracting {z}...")
        shutil.unpack_archive(os.path.join(drive_path, z), temp_extract)

        # SMART ROUTING: Extract and route images to correct split
        is_explicit_test_zip = "_test" in z.lower()
        for root, dirs, files in os.walk(temp_extract):
            for file in files:
                if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.webp', '.tif', '.tiff')):
                    path_lower = root.lower()
                    if 'test' in path_lower or 'val' in path_lower or is_explicit_test_zip:
                        split = 'test'
                    else:
                        split = 'train'

                    final_class_dir = os.path.join(LOCAL_TRAIN_DIR, split, class_folder)
                    os.makedirs(final_class_dir, exist_ok=True)

                    # Rename image to prevent overwriting images with same name from other datasets
                    new_name = f"{zip_basename}_{file}"
                    shutil.move(os.path.join(root, file), os.path.join(final_class_dir, new_name))

        shutil.rmtree(temp_extract)

if not found_zips:
    raise FileNotFoundError("❌ No zip files found on Drive.")

print("\n2. Loading Data into HuggingFace Dataset...")
raw_dataset = load_dataset("imagefolder", data_dir=LOCAL_TRAIN_DIR)

global_labels = ['clean', 'rust', 'crack']
label2id = {c: i for i, c in enumerate(global_labels)}
id2label = {i: c for i, c in enumerate(global_labels)}

folder_names = raw_dataset['train'].features['label'].names
folder2id = {idx: label2id['crack' if name == 'cracks' else name] for idx, name in enumerate(folder_names)}

train_ds = raw_dataset['train'].map(lambda x: {'label': folder2id[x['label']]})
test_ds = raw_dataset['test'].map(lambda x: {'label': folder2id[x['label']]})

# ==========================================
# CLASS LIMITING
# ==========================================
print("\n3. Processing Full Dataset Distribution...")
CLASS_LIMITS = {'clean': 10000, 'rust': None, 'crack': None}

def apply_limits(ds, limits):
    labels_array = np.array(ds['label'])
    indices_to_keep = []
    for name, lbl_id in label2id.items():
        class_indices = np.where(labels_array == lbl_id)[0]
        np.random.seed(42)
        np.random.shuffle(class_indices)
        limit = limits.get(name)
        if limit is not None and len(class_indices) > limit:
            class_indices = class_indices[:limit]
        indices_to_keep.extend(class_indices.tolist())
    np.random.shuffle(indices_to_keep)
    return ds.select(indices_to_keep)

train_ds = apply_limits(train_ds, CLASS_LIMITS)

# Counts
rust_count = sum(1 for x in train_ds['label'] if x == label2id['rust'])
crack_count = sum(1 for x in train_ds['label'] if x == label2id['crack'])
clean_count = sum(1 for x in train_ds['label'] if x == label2id['clean'])
total_images = len(train_ds)

print(f"   - Rust:  {rust_count}")
print(f"   - Crack: {crack_count}")
print(f"   - Clean: {clean_count}")

print("\n4. Initializing Model & Image Processor...")
model_checkpoint = "google/vit-base-patch16-224-in21k"
image_processor = AutoImageProcessor.from_pretrained(model_checkpoint, use_fast=True)

# ==========================================
# ADVANCED AUGMENTATION (Anti-Rust/Crack Confusion)
# ==========================================
size = image_processor.size.get("shortest_edge", image_processor.size.get("height", 224))
normalize = Normalize(mean=image_processor.image_mean, std=image_processor.image_std)

train_transforms = Compose([
    RandomResizedCrop(size, scale=(0.8, 1.0)),
    RandomHorizontalFlip(),
    RandomVerticalFlip(),
    RandomRotation(15),
    ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    # Drops color 20% of the time, forcing AI to learn geometric structure (cracks) over color (rust)
    RandomGrayscale(p=0.2),
    # Sharpens the image 50% of the time to make hairline cracks stand out against background textures
    RandomAdjustSharpness(sharpness_factor=2, p=0.5),
    ToTensor(),
    normalize,
])

test_transforms = Compose([
    Resize(size),
    CenterCrop(size),
    ToTensor(),
    normalize,
])

def preprocess_train(batch):
    batch["pixel_values"] = [train_transforms(img.convert("RGB")) for img in batch["image"]]
    batch["labels"] = batch["label"]
    return batch

def preprocess_test(batch):
    batch["pixel_values"] = [test_transforms(img.convert("RGB")) for img in batch["image"]]
    batch["labels"] = batch["label"]
    return batch

prepared_train = train_ds.with_transform(preprocess_train)
prepared_test = test_ds.with_transform(preprocess_test)

# ==========================================
# ADVANCED LOSS HANDLING: FOCAL LOSS
# ==========================================
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.weight = weight
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.weight, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

weights = torch.tensor([
    math.sqrt(total_images / clean_count) if clean_count > 0 else 1.0,
    math.sqrt(total_images / rust_count) if rust_count > 0 else 1.0,
    math.sqrt(total_images / crack_count) if crack_count > 0 else 1.0
], dtype=torch.float)

class ImbalancedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fct = FocalLoss(weight=self.class_weights.to(outputs.logits.device), gamma=2.0)
        loss = loss_fct(outputs.logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# ==========================================
# DYNAMIC HARDWARE OPTIMIZATION
# ==========================================
print("\n5. Detecting Compute Hardware & Optimizing...")
has_gpu = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if has_gpu else "CPU"

# Defaults
train_batch_size = 8
num_workers = 2
prefetch = 2
use_tf32 = False

if has_gpu:
    # 1. Enable cuDNN benchmarking for optimal hardware-specific algorithms
    torch.backends.cudnn.benchmark = True

    # 2. Check VRAM and scale Batch Size dynamically
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    if vram_gb > 70:      # 80GB VRAM (e.g., A100 80GB)
        train_batch_size = 512
    elif vram_gb > 35:    # 40GB VRAM (e.g., A100 40GB)
        train_batch_size = 256
    elif vram_gb > 14:    # 16GB VRAM (e.g., T4, V100)
        train_batch_size = 128
    else:                 # Low VRAM
        train_batch_size = 64

    # 3. Check System RAM & CPU cores to scale Data Loaders
    sys_ram_gb = psutil.virtual_memory().total / (1024**3)
    cpu_cores = multiprocessing.cpu_count()

    if sys_ram_gb > 100:
        num_workers = min(cpu_cores, 16) # Extremely high RAM, saturate CPUs
        prefetch = 4
    elif sys_ram_gb > 30:
        num_workers = min(cpu_cores, 8)
        prefetch = 3
    else:
        num_workers = min(cpu_cores, 4)
        prefetch = 2

    # 4. Enable TF32 for Ampere architecture (A100/A10G)
    use_tf32 = torch.cuda.get_device_capability()[0] >= 8

training_args = TrainingArguments(
    output_dir="./defect_model_checkpoints",
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    weight_decay=0.05,
    label_smoothing_factor=0.1,
    per_device_train_batch_size=train_batch_size,
    per_device_eval_batch_size=train_batch_size,
    fp16=has_gpu,
    tf32=use_tf32,
    torch_compile=False,
    optim="adamw_torch_fused" if has_gpu else "adamw_torch",
    dataloader_num_workers=num_workers,
    dataloader_prefetch_factor=prefetch if num_workers > 0 else None,
    dataloader_pin_memory=has_gpu,
    num_train_epochs=4,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)

trainer = ImbalancedTrainer(
    class_weights=weights,
    model=AutoModelForImageClassification.from_pretrained(model_checkpoint, label2id=label2id, id2label=id2label, ignore_mismatched_sizes=True),
    args=training_args,
    train_dataset=prepared_train,
    eval_dataset=prepared_test,
    data_collator=lambda b: {'pixel_values': torch.stack([x['pixel_values'] for x in b]), 'labels': torch.tensor([x['labels'] for x in b])},
    compute_metrics=lambda p: evaluate.load("accuracy").compute(predictions=np.argmax(p.predictions, axis=1), references=p.label_ids),
)

print("\n" + "="*50)
print(f"🚀 READY TO TRAIN ON {gpu_name.upper()}!")
print(f"-> Selected Batch Size: {train_batch_size} (VRAM: {vram_gb:.1f} GB)")
print(f"-> Selected CPU Workers: {num_workers} (RAM: {sys_ram_gb:.1f} GB)")
print(f"-> Prefetch Factor: {prefetch} | TF32: {use_tf32}")
print(f"-> Full Training Dataset Size: {total_images} images")
print("="*50 + "\n")

input("Press [ENTER] to start the training loop...")

trainer.train()

# Final Save to Drive
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
final_model_path = os.path.join(DRIVE_MODEL_DIR, "rust_and_crack_model_final")

trainer.save_model(final_model_path)
image_processor.save_pretrained(final_model_path)

print("✅ Training complete. Model and Processor saved to Drive.")

Installing required AI libraries...

1. Importing Modular Data from Google Drive Class Folders...
   [~] Extracting NikitaRust_rust_train.zip...
   [~] Extracting NikitaRust_rust_test.zip...
   [~] Extracting Francesco_rust_train.zip...
   [~] Extracting Francesco_rust_test.zip...
   [~] Extracting BenPepper_rust_train.zip...
   [~] Extracting BenPepper_rust_test.zip...
   [~] Extracting CorrSave_rust_train.zip...
   [~] Extracting CorrSave_rust_test.zip...
   [~] Extracting CorrSaveRust_rust_test.zip...
   [~] Extracting CorrSaveRust_rust_train.zip...
   [~] Extracting BinKhoaLeRust_rust_test.zip...
   [~] Extracting BinKhoaLeRust_rust_train.zip...
   [~] Extracting BinKhoaLe_rust_train.zip...
   [~] Extracting BinKhoaLe_rust_test.zip...
   [~] Extracting roboflow_test.zip...
   [~] Extracting roboflow_train.zip...
   [~] Extracting ConcreteCrack_cracks_test.zip...
   [~] Extracting ConcreteCrack_cracks_train.zip...
   [~] Extracting ECC_cracks_train.zip...
   [~] Extracting ECC_crack

Resolving data files:   0%|          | 0/47110 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/8806 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Training complete. Model and Processor saved to Drive.


In [ ]:
#@title Viewer (Colab GPU Optimized - Direct Inference w/ Benchmarking)
# ==============================================================================
# CELL 3: MULTI-DEFECT INFERENCE & BENCHMARKING SCRIPT
# Uses direct PyTorch tensor processing for maximum inference speed.
# ==============================================================================

import os
import time
import io
import platform
import csv
import gc

# --- INSTALL REQUIRED LIBRARIES ---
print("Checking and installing required AI libraries...")
os.system("pip install -q transformers pillow torch matplotlib psutil opencv-python-headless")

import psutil
import torch
import torch.nn.functional as F
import cv2
import numpy as np
from PIL import Image, ImageDraw
from transformers import AutoImageProcessor, AutoModelForImageClassification
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

# --- CONNECT GOOGLE DRIVE ---
try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        print("Mounting Google Drive...")
        drive.mount('/content/drive')
except:
    print("Not in Colab environment. Skipping Drive mount.")

# --- CONFIGURATION ---
MODEL_PATH = "/content/drive/MyDrive/Colab Notebooks/Capstone/models/rust_and_crack_model_final"

MEDIA_TYPE = "Video" # @param ["Photo", "Video"]
MAX_VIDEO_FPS = 30 # @param {"type":"slider","min":10,"max":60,"step":5}
STATISTICS_MODE = True # @param ["True","False"] {"type":"raw"}
DEEP_SEARCH_MODE = True # @param ["True","False"] {"type":"raw"}
GRID_SIZE = 5 # @param {"type":"slider","min":1,"max":20,"step":1}
MIN_CONFIDENCE = 80 # @param {"type":"slider","min":50,"max":100,"step":5}
MAX_IMAGE_SIZE = 1024
WEIGHT_FACTOR = 0.1

# --- HARDWARE PROFILER ---
def print_hardware_profile():
    """Extracts and prints the current system's CPU, RAM, and GPU specs."""
    print("\n" + "="*50)
    print("🖥️  HARDWARE PROFILE (BENCHMARK ENVIRONMENT)")
    print("="*50)

    print(f"OS:         {platform.system()} {platform.release()} ({platform.machine()})")
    ram_gb = psutil.virtual_memory().total / (1024**3)
    print(f"System RAM: {ram_gb:.2f} GB")

    physical_cores = psutil.cpu_count(logical=False)
    logical_cores = psutil.cpu_count(logical=True)
    print(f"CPU Cores:  {physical_cores} Physical / {logical_cores} Logical")

    sys_gpu_name = "CPU Only"
    if torch.cuda.is_available():
        sys_gpu_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"GPU:        {sys_gpu_name}")
        print(f"GPU VRAM:   {vram_gb:.2f} GB")
        print(f"CUDA Cap:   {torch.cuda.get_device_capability(0)}")
    else:
        print("GPU:        None Detected (Running on CPU)")

    print("="*50 + "\n")
    return sys_gpu_name

print(f"Loading the fine-tuned detector from: {MODEL_PATH}")

try:
    # --- GPU/CPU OPTIMIZATIONS ---
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32

    if device.type == 'cpu':
        print("⚠️ No GPU detected. Running in CPU-ONLY mode. (Expect slower inference times)")
    else:
        print(f"✅ GPU detected. Running on {torch.cuda.get_device_name(0)}")

        # MODERN OPTIMIZATION 1: CuDNN Auto-Tuning
        torch.backends.cudnn.benchmark = True

    # --- ROBUST IMAGE PROCESSOR LOADING ---
    try:
        processor = AutoImageProcessor.from_pretrained(MODEL_PATH)
    except Exception:
        print("Local image processor config not found. Falling back to base google/vit config...")
        processor = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224-in21k")

    # --- DIRECT MODEL INITIALIZATION ---
    model = AutoModelForImageClassification.from_pretrained(MODEL_PATH)
    model.to(device, dtype=dtype)
    model.eval()

    # MODERN OPTIMIZATION 2: PyTorch 2.0 Graph Compilation
    if device.type == 'cuda':
        print("⚙️ Applying PyTorch 2.0 Graph Compilation (Warmup will take slightly longer, inference will be faster)...")
        try:
            model = torch.compile(model, dynamic=True)
        except Exception as e:
            print(f"⚠️ torch.compile failed (falling back to standard eager execution): {e}")

    label_dict = {k.lower(): v for k, v in model.config.label2id.items()}
    rust_idx = label_dict.get('rust', 1)
    crack_idx = label_dict.get('crack', 2)
    clean_idx = label_dict.get('clean', 0)

    print(f"Model loaded successfully on {str(device).upper()} ({dtype})!")
except Exception as e:
    print(f"❌ Error loading model: {e}")
    print(f"Check if the model exists at: {MODEL_PATH}")


def extract_true_label(filename):
    """Parses filename prefix to determine the ground truth label."""
    fname = filename.lower()
    if fname.startswith("corrosion_") or fname.startswith("rust_"):
        return "RUST"
    elif fname.startswith("crack_"):
        return "CRACK"
    elif fname.startswith("clean_"):
        return "CLEAN"
    return "UNKNOWN"


def analyze_image(img, grid_size, deep_search, display_img=True, return_img=False, true_label="UNKNOWN"):
    """Analyzes a pre-loaded PIL Image and returns stats. Returns the overlaid image if return_img=True."""
    try:
        start_time = time.time()
        img = img.convert("RGBA") # Enforce RGBA for proper overlay blending
        width, height = img.size

        grids_to_run = []
        if deep_search:
            g = grid_size
            while g > 0:
                grids_to_run.append(g)
                g -= 2
        else:
            grids_to_run = [grid_size]

        crops = []
        crop_metadata = []

        for current_grid in grids_to_run:
            step_x = width // current_grid
            step_y = height // current_grid

            for row in range(current_grid):
                for col in range(current_grid):
                    left = col * step_x
                    top = row * step_y
                    right = width if col == current_grid - 1 else (col + 1) * step_x
                    bottom = height if row == current_grid - 1 else (row + 1) * step_y

                    cropped_img = img.crop((left, top, right, bottom)).convert("RGB")
                    crops.append(cropped_img)
                    crop_metadata.append((current_grid, left, top, right, bottom))

        # Tensor Inference
        inputs = processor(images=crops, return_tensors="pt").to(device, dtype=dtype)

        # MODERN OPTIMIZATION 3: Mixed Precision Autocast
        with torch.inference_mode(), torch.autocast(device_type=device.type, dtype=dtype):
            outputs = model(**inputs)
            probabilities = F.softmax(outputs.logits, dim=-1)

        # Spatial Averaging
        base_step_x = width // grid_size
        base_step_y = height // grid_size
        base_grid_scores = {}

        for row in range(grid_size):
            for col in range(grid_size):
                left = col * base_step_x
                top = row * base_step_y
                right = width if col == grid_size - 1 else (col + 1) * base_step_x
                bottom = height if row == grid_size - 1 else (row + 1) * base_step_y

                cx = left + (right - left) / 2
                cy = top + (bottom - top) / 2

                base_grid_scores[(row, col)] = {
                    'rust': 0.0, 'crack': 0.0, 'clean': 0.0,
                    'total_weight': 0.0, 'coords': (left, top, right, bottom)
                }

                for i, (g_size, c_left, c_top, c_right, c_bottom) in enumerate(crop_metadata):
                    if c_left <= cx <= c_right and c_top <= cy <= c_bottom:
                        weight = 1.0 + ((grid_size - g_size) * WEIGHT_FACTOR)
                        base_grid_scores[(row, col)]['rust'] += (probabilities[i][rust_idx].item() * 100) * weight
                        base_grid_scores[(row, col)]['crack'] += (probabilities[i][crack_idx].item() * 100) * weight
                        base_grid_scores[(row, col)]['clean'] += (probabilities[i][clean_idx].item() * 100) * weight
                        base_grid_scores[(row, col)]['total_weight'] += weight

        # Verdict logic
        max_rust_conf = 0.0
        max_crack_conf = 0.0
        defects_found = set()

        if display_img or return_img:
            overlay = Image.new('RGBA', img.size, (255, 255, 255, 0))
            draw = ImageDraw.Draw(overlay)
            line_width = max(1, width // 500)

        for (row, col), data in base_grid_scores.items():
            total_w = data['total_weight']
            avg_rust = data['rust'] / total_w
            avg_crack = data['crack'] / total_w
            left, top, right, bottom = data['coords']

            max_rust_conf = max(max_rust_conf, avg_rust)
            max_crack_conf = max(max_crack_conf, avg_crack)
            combined_defect_conf = avg_rust + avg_crack

            if combined_defect_conf >= MIN_CONFIDENCE:
                if avg_rust >= 30.0 and avg_crack >= 30.0:
                    defects_found.update(["RUST", "CRACK"])
                elif avg_rust > avg_crack:
                    defects_found.add("RUST")
                else:
                    defects_found.add("CRACK")

                if display_img or return_img:
                    r_val = int((avg_rust / combined_defect_conf) * 255)
                    b_val = int((avg_crack / combined_defect_conf) * 255)
                    outline_color = (r_val, 0, b_val, 255)
                    fill_opacity = min(130, int((combined_defect_conf / 100.0) * 180))
                    fill_color = (r_val, 0, b_val, fill_opacity)
                    draw.rectangle([left, top, right, bottom], outline=outline_color, fill=fill_color, width=line_width)

        # Determine overall prediction
        if not defects_found:
            prediction = "CLEAN"
        elif "RUST" in defects_found and "CRACK" in defects_found:
            prediction = "RUST" if max_rust_conf > max_crack_conf else "CRACK"
        else:
            prediction = list(defects_found)[0]

        is_correct = (prediction == true_label) if true_label != "UNKNOWN" else None
        process_time = time.time() - start_time

        # Create overlay image if requested
        if display_img or return_img:
            final_img = Image.alpha_composite(img, overlay).convert("RGB")

        if display_img:
            print(f"\n--- Visual Analysis for Mode: Grid {grid_size}x{grid_size} | Deep Search: {deep_search} ---")
            display(final_img.resize((700, int(700 * height / width))))

            print("-" * 45)
            if defects_found:
                print(f"STATUS: ⚠️ ACTION REQUIRED")
                print(f"DEFECTS: {' & '.join(sorted(list(defects_found)))} DETECTED")
            else:
                print(f"STATUS: ✅ NOMINAL (CLEAN)")

            if true_label != "UNKNOWN":
                pass_fail = "✅ PASS" if is_correct else f"❌ FAIL (Predicted: {prediction}, Actual: {true_label})"
                print(f"SELF-CHECK: {pass_fail}")
            print("="*45 + "\n")

        if return_img:
            return is_correct, process_time, final_img

        return is_correct, process_time

    except Exception as e:
        print(f"Error during analysis: {e}")
        if return_img:
            return False, 0.0, None
        return False, 0.0

# ==============================================================================
# INTERACTIVE / STATISTICS LOOP
# ==============================================================================
try:
    from google.colab import files

    print("-" * 50)
    print("Upload media for inspection.")
    print("Note: Name files starting with 'corrosion_', 'crack_', or 'clean_' for accuracy tracking.")
    uploaded = files.upload()

    if uploaded:
        if MEDIA_TYPE == "Video":
            # --- VIDEO MODE ---
            clear_output(wait=True)
            gpu_name = print_hardware_profile()

            for filename, data in uploaded.items():
                print(f"🎬 Processing Video: {filename}")

                # Save bytes to a temporary file for OpenCV
                temp_in = f"temp_{filename}"
                with open(temp_in, "wb") as f:
                    f.write(data)

                cap = cv2.VideoCapture(temp_in)
                original_fps = cap.get(cv2.CAP_PROP_FPS)
                if original_fps == 0 or original_fps is None:
                    original_fps = 30.0 # Fallback

                width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
                height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
                total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

                if total_frames == 0:
                    print("❌ Could not read video frames. Skipping...")
                    os.remove(temp_in)
                    continue

                # --- FPS DOWNSAMPLING LOGIC ---
                frame_skip = max(1, int(round(original_fps / MAX_VIDEO_FPS)))
                effective_fps = original_fps / frame_skip
                expected_out_frames = total_frames // frame_skip

                # Adjust scale to keep processing fast
                scale = 1.0
                if max(width, height) > MAX_IMAGE_SIZE:
                    scale = MAX_IMAGE_SIZE / max(width, height)

                out_width = int(width * scale)
                out_height = int(height * scale)

                out_filename = f"analyzed_{filename}"
                if not out_filename.lower().endswith('.mp4'):
                    out_filename = out_filename.rsplit('.', 1)[0] + '.mp4'

                # Setup video writer
                fourcc = cv2.VideoWriter_fourcc(*'mp4v')
                out = cv2.VideoWriter(out_filename, fourcc, effective_fps, (out_width, out_height))

                print(f"Original Video: {total_frames} frames @ {original_fps:.2f} FPS")
                print(f"Target Output:  {expected_out_frames} frames @ {effective_fps:.2f} FPS (Skipping every {frame_skip} frames)")
                print(f"Target Res:     {out_width}x{out_height}")
                print(f"Applying Model: Grid {GRID_SIZE}x{GRID_SIZE} | Deep Search: {DEEP_SEARCH_MODE}\n")

                # --- WARMUP PHASE ---
                print("🔥 Warming up PyTorch Compiler for Video Mode... (Please wait)")
                dummy_img = Image.new('RGBA', (out_width, out_height), color=(128, 128, 128, 255))
                analyze_image(dummy_img, GRID_SIZE, DEEP_SEARCH_MODE, display_img=False, return_img=False, true_label="UNKNOWN")
                if torch.cuda.is_available():
                    torch.cuda.synchronize()
                print("✅ Warmup complete. Processing actual frames...\n")

                raw_frame_count = 0
                processed_frame_count = 0

                # Tracking variables for true instantaneous FPS
                start_vid_time = time.time()
                last_time = start_vid_time
                last_frame_count = 0

                while cap.isOpened():
                    ret, frame = cap.read()
                    if not ret:
                        break

                    raw_frame_count += 1

                    # --- OPTIMIZATION: Skip frames to reach target FPS ---
                    if raw_frame_count % frame_skip != 0:
                        continue

                    # Resize quickly using OpenCV before converting to PIL
                    if scale < 1.0:
                        frame = cv2.resize(frame, (out_width, out_height), interpolation=cv2.INTER_LINEAR)

                    # Convert cv2 BGR to PIL RGB
                    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    pil_img = Image.fromarray(frame_rgb)

                    # Analyze and extract overlaid frame
                    _, _, final_frame_img = analyze_image(pil_img, GRID_SIZE, DEEP_SEARCH_MODE, display_img=False, return_img=True, true_label="UNKNOWN")

                    # Convert overlaid PIL RGB back to cv2 BGR
                    res_frame = cv2.cvtColor(np.array(final_frame_img), cv2.COLOR_RGB2BGR)
                    out.write(res_frame)

                    processed_frame_count += 1

                    # --- OPTIMIZATION: VRAM Garbage Collection for Lower-End GPUs ---
                    if processed_frame_count % 30 == 0:
                        if torch.cuda.is_available():
                            torch.cuda.empty_cache() # Flush memory fragmentation
                        gc.collect() # Force Python to dump orphaned RAM arrays

                    # --- OPTIMIZATION: Explicit deletion to prevent RAM build-up ---
                    del frame, frame_rgb, pil_img, final_frame_img, res_frame

                    if processed_frame_count % 10 == 0 or processed_frame_count == expected_out_frames:
                        # Calculate Average FPS (Total Time)
                        elapsed_total = time.time() - start_vid_time
                        fps_avg = processed_frame_count / elapsed_total

                        # Calculate Instantaneous FPS (Time since last print block)
                        elapsed_recent = time.time() - last_time
                        frames_recent = processed_frame_count - last_frame_count
                        fps_instant = frames_recent / elapsed_recent if elapsed_recent > 0 else 0

                        print(f"   Processed {processed_frame_count}/{expected_out_frames} frames | Instant Speed: {fps_instant:.1f} fps | Avg Speed: {fps_avg:.1f} fps")

                        # Reset instant trackers
                        last_time = time.time()
                        last_frame_count = processed_frame_count

                cap.release()
                out.release()
                os.remove(temp_in)

                print(f"\n✅ Video processing complete in {time.time() - start_vid_time:.1f}s")
                print(f"📥 Automatically downloading: {out_filename}")
                files.download(out_filename)

        elif not STATISTICS_MODE:
            # --- STANDARD PHOTO MODE ---
            clear_output(wait=True)
            gpu_name = print_hardware_profile()
            for filename, data in uploaded.items():
                print(f"Processing: {filename}")
                true_lbl = extract_true_label(filename)

                # Pre-process image for standard mode
                img = Image.open(io.BytesIO(data)).convert("RGBA")
                if max(img.size) > MAX_IMAGE_SIZE:
                    img.thumbnail((MAX_IMAGE_SIZE, MAX_IMAGE_SIZE), Image.Resampling.BILINEAR)

                analyze_image(img, GRID_SIZE, DEEP_SEARCH_MODE, display_img=True, return_img=False, true_label=true_lbl)

        else:
            # --- STATISTICS PHOTO MODE ---
            clear_output(wait=True)
            print("🚀 RUNNING STATISTICS MODE BENCHMARK...")
            gpu_name = print_hardware_profile()

            # --- PRE-LOAD DATASET INTO RAM ---
            print("⚙️ Pre-processing dataset into RAM to eliminate I/O bottleneck...")
            preloaded_dataset = []

            for filename, data in uploaded.items():
                true_lbl = extract_true_label(filename)
                if true_lbl == "UNKNOWN":
                    continue

                # Decode, Resize, and Cache to RAM exactly once
                img = Image.open(io.BytesIO(data)).convert("RGBA")
                if max(img.size) > MAX_IMAGE_SIZE:
                    img.thumbnail((MAX_IMAGE_SIZE, MAX_IMAGE_SIZE), Image.Resampling.BILINEAR)

                preloaded_dataset.append((filename, img, true_lbl))

            eval_images = len(preloaded_dataset)
            if eval_images == 0:
                print("ERROR: No properly prefixed files found. Cannot calculate stats.")
            else:
                print(f"✅ Successfully loaded {eval_images} valid images into high-speed memory.\n")

                # Use the configured GRID_SIZE as the maximum boundary for benchmarking
                grid_sizes = list(range(1, GRID_SIZE + 1)) # 1x1 up to GRID_SIZE x GRID_SIZE

                print("🔥 Warming up hardware and compiling computational graphs...")
                print("   (This may take a minute as PyTorch optimizes for varying batch sizes)")
                dummy_img = Image.new('RGBA', (MAX_IMAGE_SIZE, MAX_IMAGE_SIZE), color=(128, 128, 128, 255))

                # Run a silent pass for ALL grid sizes to compile the graph for every possible crop batch
                for w_size in grid_sizes:
                    analyze_image(dummy_img, w_size, False, return_img=False, true_label="UNKNOWN")

                if torch.cuda.is_available():
                    torch.cuda.synchronize() # Wait for GPU to finish compiling and initializing
                print("✅ Warmup & Compilation complete.\n")

                modes = {"Normal": False, "Deep Search": True}

                stats = {
                    "Normal": {"accuracy": [], "ips": [], "cpu_usage": [], "gpu_vram": []},
                    "Deep Search": {"accuracy": [], "ips": [], "cpu_usage": [], "gpu_vram": []}
                }

                for mode_name, is_deep in modes.items():
                    print(f"\n--- Testing {mode_name} Mode ---")

                    for g_size in grid_sizes:
                        correct_count = 0
                        total_time = 0.0

                        # Prime CPU Tracker and reset GPU Peak Memory Tracker
                        psutil.cpu_percent(interval=None)
                        if torch.cuda.is_available():
                            torch.cuda.reset_peak_memory_stats()

                        # Run via pure RAM preloaded dataset
                        for filename, pre_img, true_lbl in preloaded_dataset:
                            is_corr, p_time = analyze_image(pre_img, g_size, is_deep, display_img=False, return_img=False, true_label=true_lbl)

                            if is_corr: correct_count += 1
                            total_time += p_time

                        # Capture System Load over that batch
                        avg_cpu = psutil.cpu_percent(interval=None)
                        peak_vram_mb = (torch.cuda.max_memory_allocated() / (1024**2)) if torch.cuda.is_available() else 0.0

                        # Calculate Stats
                        acc = (correct_count / eval_images) * 100
                        avg_time = total_time / eval_images
                        ips = 1.0 / avg_time if avg_time > 0 else 0

                        stats[mode_name]["accuracy"].append(acc)
                        stats[mode_name]["ips"].append(ips)
                        stats[mode_name]["cpu_usage"].append(avg_cpu)
                        stats[mode_name]["gpu_vram"].append(peak_vram_mb)

                        print(f"Grid {g_size}x{g_size} -> Acc: {acc:.1f}% | IPS: {ips:.2f} | CPU: {avg_cpu}% | VRAM: {peak_vram_mb:.1f} MB")

                # --- PLOTTING AND SAVING MULTI-CHART STATISTICS ---
                if len(stats["Normal"]["accuracy"]) > 0:
                    # 1. Get Model Modification Date
                    try:
                        mtime = os.path.getmtime(MODEL_PATH)
                        model_mod_date = time.strftime("%Y-%m-%d_%H-%M", time.localtime(mtime))
                    except OSError:
                        model_mod_date = "unknown_date"

                    # 2. Sanitize Hardware Name
                    safe_gpu_name = gpu_name.replace("/", "_").replace("\\", "_").replace(":", "").replace(" ", "_")

                    # 3. Create Nested Directory structure: Benchmarks -> Hardware Type
                    BENCHMARK_DIR = f"/content/drive/MyDrive/Colab Notebooks/Capstone/Benchmarks/{safe_gpu_name}"
                    os.makedirs(BENCHMARK_DIR, exist_ok=True)

                    # ================= CHART 1: ACCURACY VS IPS =================
                    fig1, ax1 = plt.subplots(figsize=(10, 6))
                    fig1.suptitle(f"Model Inference Benchmarks (Hardware: {gpu_name})", fontsize=14, fontweight='bold')

                    color1 = 'tab:blue'
                    ax1.set_xlabel('Grid Size (NxN)', fontweight='bold')
                    ax1.set_ylabel('Accuracy (%)', color=color1, fontweight='bold')
                    ax1.plot(grid_sizes, stats["Normal"]["accuracy"], marker='o', linestyle='-', color='lightblue', label='Normal Acc')
                    ax1.plot(grid_sizes, stats["Deep Search"]["accuracy"], marker='s', linestyle='-', color='blue', label='Deep Search Acc')
                    ax1.tick_params(axis='y', labelcolor=color1)
                    ax1.set_xticks(grid_sizes)
                    ax1.set_ylim(0, 105)

                    ax2 = ax1.twinx()
                    color2 = 'tab:red'
                    ax2.set_ylabel('Images / Second (IPS)', color=color2, fontweight='bold')
                    ax2.plot(grid_sizes, stats["Normal"]["ips"], marker='^', linestyle='--', color='lightcoral', label='Normal IPS')
                    ax2.plot(grid_sizes, stats["Deep Search"]["ips"], marker='v', linestyle='--', color='darkred', label='Deep Search IPS')
                    ax2.tick_params(axis='y', labelcolor=color2)

                    ax1.set_title("Performance & Speed Scaling")
                    lines_1, labels_1 = ax1.get_legend_handles_labels()
                    lines_2, labels_2 = ax2.get_legend_handles_labels()
                    ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='center left')
                    ax1.grid(True, alpha=0.3)
                    fig1.tight_layout()

                    # Save Chart 1 to Drive
                    plot1_path = os.path.join(BENCHMARK_DIR, f"Performance_Speed_{model_mod_date}.png")
                    fig1.savefig(plot1_path, bbox_inches='tight')
                    print(f"✅ Saved chart to: {plot1_path}")

                    # ================= CHART 2: CPU VS GPU USAGE =================
                    fig2, ax3 = plt.subplots(figsize=(10, 6))
                    fig2.suptitle(f"Model Inference Benchmarks (Hardware: {gpu_name})", fontsize=14, fontweight='bold')

                    color3 = 'tab:green'
                    ax3.set_xlabel('Grid Size (NxN)', fontweight='bold')
                    ax3.set_ylabel('Average CPU Utilization (%)', color=color3, fontweight='bold')
                    ax3.plot(grid_sizes, stats["Normal"]["cpu_usage"], marker='o', linestyle='-', color='lightgreen', label='Normal CPU %')
                    ax3.plot(grid_sizes, stats["Deep Search"]["cpu_usage"], marker='s', linestyle='-', color='darkgreen', label='Deep Search CPU %')
                    ax3.tick_params(axis='y', labelcolor=color3)
                    ax3.set_xticks(grid_sizes)
                    ax3.set_ylim(0, 105)

                    ax4 = ax3.twinx()
                    color4 = 'tab:purple'
                    vram_label = 'Peak GPU VRAM Allocated (MB)' if torch.cuda.is_available() else 'No GPU Detected'
                    ax4.set_ylabel(vram_label, color=color4, fontweight='bold')
                    ax4.plot(grid_sizes, stats["Normal"]["gpu_vram"], marker='^', linestyle='--', color='plum', label='Normal VRAM')
                    ax4.plot(grid_sizes, stats["Deep Search"]["gpu_vram"], marker='v', linestyle='--', color='purple', label='Deep Search VRAM')
                    ax4.tick_params(axis='y', labelcolor=color4)

                    # Start VRAM Y-axis at 0 to show true scale
                    ax4.set_ylim(ymin=0)

                    ax3.set_title("Hardware Resource Scaling")
                    lines_3, labels_3 = ax3.get_legend_handles_labels()
                    lines_4, labels_4 = ax4.get_legend_handles_labels()
                    ax3.legend(lines_3 + lines_4, labels_3 + labels_4, loc='upper left')
                    ax3.grid(True, alpha=0.3)
                    fig2.tight_layout()

                    # Save Chart 2 to Drive
                    plot2_path = os.path.join(BENCHMARK_DIR, f"Hardware_Resource_{model_mod_date}.png")
                    fig2.savefig(plot2_path, bbox_inches='tight')
                    print(f"✅ Saved chart to: {plot2_path}")

                    # ================= EXPORT TO CSV =================
                    csv_path = os.path.join(BENCHMARK_DIR, f"Benchmark_Data_{model_mod_date}.csv")

                    # Gather system properties for flat CSV
                    sys_os = f"{platform.system()} {platform.release()}"
                    sys_ram = f"{psutil.virtual_memory().total / (1024**3):.2f}"
                    sys_cpu = f"{psutil.cpu_count(logical=True)}"
                    sys_vram_gb = f"{torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f}" if torch.cuda.is_available() else "N/A"
                    sys_cuda = f"{torch.cuda.get_device_capability(0)[0]}.{torch.cuda.get_device_capability(0)[1]}" if torch.cuda.is_available() else "N/A"

                    with open(csv_path, mode='w', newline='') as csv_file:
                        writer = csv.writer(csv_file)
                        # Write Header
                        writer.writerow([
                            "Hardware GPU", "OS", "System RAM (GB)", "CPU Cores", "Total GPU VRAM (GB)", "CUDA Cap",
                            "Mode", "Grid Size (NxN)", "Accuracy (%)", "IPS", "CPU Usage (%)", "Peak VRAM (MB)"
                        ])

                        # Write Data
                        for mode_name in modes.keys():
                            for i, g_size in enumerate(grid_sizes):
                                writer.writerow([
                                    gpu_name, sys_os, sys_ram, sys_cpu, sys_vram_gb, sys_cuda,
                                    mode_name,
                                    f"{g_size}x{g_size}",
                                    f'{stats[mode_name]["accuracy"][i]:.2f}',
                                    f'{stats[mode_name]["ips"][i]:.2f}',
                                    f'{stats[mode_name]["cpu_usage"][i]:.2f}',
                                    f'{stats[mode_name]["gpu_vram"][i]:.2f}'
                                ])
                    print(f"✅ Saved numerical data to: {csv_path}")

                    plt.show()

    else:
        print("Upload cancelled.")

except ImportError:
    print("Google Colab modules not found.")


🖥️  HARDWARE PROFILE (BENCHMARK ENVIRONMENT)
OS:         Linux 6.6.113+ (x86_64)
System RAM: 12.67 GB
CPU Cores:  1 Physical / 2 Logical
GPU:        Tesla T4
GPU VRAM:   14.56 GB
CUDA Cap:   (7, 5)

🎬 Processing Video: 2026-03-06 12-30-15 (3).mp4
Original Video: 2688 frames @ 60.00 FPS
Target Output:  1344 frames @ 30.00 FPS (Skipping every 2 frames)
Target Res:     1024x576
Applying Model: Grid 5x5 | Deep Search: True

🔥 Warming up PyTorch Compiler for Video Mode... (Please wait)
✅ Warmup complete. Processing actual frames...

   Processed 10/1344 frames | Instant Speed: 6.1 fps | Avg Speed: 6.1 fps
   Processed 20/1344 frames | Instant Speed: 5.0 fps | Avg Speed: 5.5 fps
   Processed 30/1344 frames | Instant Speed: 4.3 fps | Avg Speed: 5.0 fps
   Processed 40/1344 frames | Instant Speed: 6.2 fps | Avg Speed: 5.3 fps
   Processed 50/1344 frames | Instant Speed: 6.7 fps | Avg Speed: 5.5 fps
   Processed 60/1344 frames | Instant Speed: 5.2 fps | Avg Speed: 5.4 fps
   Processed 70/1344 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>